# Handling Missing Values

Missing values are observations that were not recorded, could not be measured, or do not apply. In R, a missing value is usually represented by `NA`. Missing data need careful handling because different choices can change the size, representativeness, and conclusions of an analysis.

This chapter introduces ways to find, summarize, remove, and fill missing values using base R. The examples use a small study dataset so the steps are reproducible.

## Learning goals

By the end, you should be able to:

- Identify missing values in a vector or data frame.
- Calculate missing counts and percentages.
- Remove incomplete observations when that choice is justified.
- Apply simple numeric imputation while understanding its limitations.
- Choose a strategy based on how and why data are missing.

## Create an example dataset

The data below contain a treatment group, a numeric measurement, and an age. Some measurements and ages are missing. `NA` is R's standard marker for a missing value.

In [10]:
study <- data.frame(
  id = 1:12,
  treatment = c("Control", "Control", "Control", "Control", "Control", "Control",
                "New", "New", "New", "New", "New", "New"),
  measurement = c(12.1, 11.8, NA, 13.0, 12.5, 11.9, 14.2, NA, 15.1, 14.8, NA, 15.4),
  age = c(34, 51, 46, NA, 39, 58, 42, 37, NA, 49, 53, 44)
)
study

id,treatment,measurement,age
<int>,<chr>,<dbl>,<dbl>
1,Control,12.1,34
2,Control,11.8,51
3,Control,NA,46
4,Control,13.0,NA
5,Control,12.5,39
6,Control,11.9,58
7,New,14.2,42
8,New,NA,37
9,New,15.1,NA


## Detect missing values

Use `is.na()` to locate missing entries. It returns `TRUE` for missing values and `FALSE` otherwise. The sum of a logical vector counts its `TRUE` values because R treats `TRUE` as 1 and `FALSE` as 0.

In [11]:
is.na(study$measurement)
sum(is.na(study$measurement))
anyNA(study)

[1] FALSE FALSE  TRUE FALSE FALSE FALSE FALSE  TRUE FALSE FALSE  TRUE FALSE

[1] 3

[1] TRUE

## Summarize missingness

Count missing entries in each column and calculate their percentages. A percentage gives useful context when columns have different numbers of observations.

In [12]:
missing_count <- colSums(is.na(study))
missing_percent <- colMeans(is.na(study)) * 100
data.frame(
  variable = names(study),
  missing_count = missing_count,
  missing_percent = round(missing_percent, 1)
)

,variable,missing_count,missing_percent
,<chr>,<dbl>,<dbl>
id,id,0,0.0
treatment,treatment,0,0.0
measurement,measurement,3,25.0
age,age,2,16.7


## Look for patterns

Missingness is not always random. Ask whether missing values are concentrated in a group, time period, measurement, or type of participant. The table below marks each missing entry by row and column. These checks do not prove why values are missing, but they can reveal patterns worth investigating.

In [13]:
is.na(study)
with(study, table(treatment, is.na(measurement)))

id,treatment,measurement,age
FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,TRUE,FALSE
FALSE,FALSE,FALSE,TRUE
FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,FALSE,FALSE
FALSE,FALSE,TRUE,FALSE
FALSE,FALSE,FALSE,TRUE
FALSE,FALSE,FALSE,FALSE


         
treatment FALSE TRUE
  Control     5    1
  New         4    2

## Why values may be missing

The reason for missingness matters:

- **MCAR (missing completely at random):** the chance of a value being missing is unrelated to observed or unobserved data.
- **MAR (missing at random, conditional on observed data):** missingness is related to information you have observed. For example, younger participants may be less likely to complete a follow-up survey.
- **MNAR (missing not at random):** missingness depends on the unobserved value itself or another unobserved factor. For example, people with very high symptom scores may be less likely to report them.

These labels describe assumptions about the data collection process. They cannot usually be determined from the observed data alone. Discuss the context with subject-matter experts.

## Remove incomplete observations

`complete.cases()` identifies rows with no missing values in the selected data. `na.omit()` removes rows that contain at least one missing value.

Complete-case analysis is easy, but it reduces sample size and can bias results when the remaining observations are not representative. Use it only when the assumptions are defensible, and report how many observations were removed.

In [14]:
complete_rows <- complete.cases(study)
sum(complete_rows)
study_complete <- study[complete_rows, ]
study_complete
nrow(study) - nrow(study_complete)

[1] 7

,id,treatment,measurement,age
,<int>,<chr>,<dbl>,<dbl>
1,1,Control,12.1,34
2,2,Control,11.8,51
5,5,Control,12.5,39
6,6,Control,11.9,58
7,7,New,14.2,42
10,10,New,14.8,49
12,12,New,15.4,44


[1] 5

## Ignore missing values in a calculation

Many R summaries return `NA` if the input contains missing values. For a quick descriptive statistic, use an available-case calculation such as `mean(x, na.rm = TRUE)`. This skips missing values for that calculation; it does not fill them in or make every analysis appropriate.

In [15]:
mean(study$measurement)
mean(study$measurement, na.rm = TRUE)
median(study$measurement, na.rm = TRUE)

[1] NA

[1] 13.42222

[1] 13

## Simple numeric imputation

Imputation replaces missing entries with estimated values. Mean or median imputation is simple, but it can distort the distribution, reduce apparent variability, and make standard errors too small. It is useful here to demonstrate the mechanics, not as an automatic default for an analysis.

The median is less sensitive to extreme values than the mean. Keep the original data unchanged so the imputed version can be compared and documented.

In [16]:
study_median <- study
measurement_median <- median(study_median$measurement, na.rm = TRUE)
study_median$measurement[is.na(study_median$measurement)] <- measurement_median
measurement_median
study_median

[1] 13

id,treatment,measurement,age
<int>,<chr>,<dbl>,<dbl>
1,Control,12.1,34
2,Control,11.8,51
3,Control,13.0,46
4,Control,13.0,NA
5,Control,12.5,39
6,Control,11.9,58
7,New,14.2,42
8,New,13.0,37
9,New,15.1,NA


## Group-wise median imputation

If groups have meaningfully different distributions, a single overall median can obscure those differences. This example fills each missing measurement with the median for its treatment group. Group-wise imputation still has limitations and should be justified by the study design and analysis plan.

In [17]:
study_group_median <- study
group_medians <- tapply(
  study_group_median$measurement,
  study_group_median$treatment,
  median,
  na.rm = TRUE
)
missing_measurement <- is.na(study_group_median$measurement)
study_group_median$measurement[missing_measurement] <-
  group_medians[study_group_median$treatment[missing_measurement]]
group_medians
study_group_median

Control     New 
  12.10   14.95

id,treatment,measurement,age
<int>,<chr>,<dbl>,<dbl>
1,Control,12.10,34
2,Control,11.80,51
3,Control,12.10,46
4,Control,13.00,NA
5,Control,12.50,39
6,Control,11.90,58
7,New,14.20,42
8,New,14.95,37
9,New,15.10,NA


## Comparing statistics before and after imputation

The first row below summarizes only the observed measurements (`NA` values are omitted). The next two rows use the same observations after filling the missing entries with the observed mean or median. Comparing rows shows that imputation can change the distribution and statistics even though no new measurements were collected.

Mean imputation puts every replacement at the observed mean, so the overall mean stays the same while the standard deviation usually decreases. Median imputation inserts values at the median; it can also change the mean and spread. The boxplot marks the inserted values with red crosses.

In [ ]:
observed <- study$measurement[!is.na(study$measurement)]
missing_count <- sum(is.na(study$measurement))
mean_fill <- mean(observed)
median_fill <- median(observed)

mean_imputed <- study$measurement
mean_imputed[is.na(mean_imputed)] <- mean_fill

median_imputed <- study$measurement
median_imputed[is.na(median_imputed)] <- median_fill

measurements <- list(
  `Observed only` = observed,
  `Mean imputed` = mean_imputed,
  `Median imputed` = median_imputed
)

comparison <- data.frame(
  method = names(measurements),
  n = vapply(measurements, length, integer(1)),
  mean = vapply(measurements, mean, numeric(1)),
  median = vapply(measurements, median, numeric(1)),
  standard_deviation = vapply(measurements, sd, numeric(1)),
  range = vapply(measurements, function(x) diff(range(x)), numeric(1))
)
comparison[-1] <- lapply(comparison[-1], round, digits = 2)
comparison

### Boxplot: observed versus imputed values

This boxplot compares the observed measurements with the mean- and median-imputed versions. Gray dots are the original observed values; red crosses show the values inserted by imputation. The imputed datasets have the same sample size, but their spread and distribution differ from the observed-only data.

In [ ]:
boxplot(
  measurements,
  outline = FALSE,
  col = c("grey90", "lightblue", "wheat"),
  ylab = "Measurement",
  main = "Observed and imputed measurements"
)

set.seed(42)
points(jitter(rep(1, length(observed)), amount = 0.08), observed,
       pch = 16, col = "grey35")
points(jitter(rep(2, length(observed)), amount = 0.08), observed,
       pch = 16, col = "grey35")
points(jitter(rep(3, length(observed)), amount = 0.08), observed,
       pch = 16, col = "grey35")
points(jitter(rep(2, missing_count), amount = 0.08), rep(mean_fill, missing_count),
       pch = 4, col = "firebrick", lwd = 2, cex = 1.3)
points(jitter(rep(3, missing_count), amount = 0.08), rep(median_fill, missing_count),
       pch = 4, col = "firebrick", lwd = 2, cex = 1.3)
legend("topleft", legend = c("Observed", "Imputed"),
       pch = c(16, 4), col = c("grey35", "firebrick"), bty = "n")

## Keep a missingness indicator

Sometimes the fact that a value is missing may itself carry information. A missingness indicator records which values were absent before imputation. It does not remove bias by itself, but it can support transparent summaries and some analysis strategies.

In [18]:
study_with_indicator <- study
study_with_indicator$measurement_was_missing <-
  is.na(study_with_indicator$measurement)
table(study_with_indicator$measurement_was_missing)


FALSE  TRUE 
    9     3 

## Practical checklist

1. Find and summarize missingness before changing the data.
2. Investigate why values are absent and whether the pattern differs across groups.
3. Choose a method appropriate to the scientific question, variable type, and assumptions.
4. Avoid replacing missing values with zero unless zero is the true measured value.
5. Keep the original data, document every transformation, and report the method.
6. For consequential analyses, consider principled approaches such as multiple imputation or models designed for incomplete data, with statistical guidance.

There is no universally correct replacement for `NA`. The best approach depends on what the missing values mean and how they arose.